# PromptOps Arena — GRPO Training Notebook

**OpenEnv India Hackathon 2026 · Team Dar3devil**

This notebook reproduces the **headline trained-agent run** end-to-end.
It can be opened directly in Google Colab (`File` → `Open notebook` → `GitHub` → paste
`https://huggingface.co/spaces/Dar3devil/promptops-arena/blob/main/notebooks/train_grpo.ipynb`)
or executed locally if you have a CUDA GPU.

## What this trains

- **Agent (trained):** `Qwen/Qwen2.5-1.5B-Instruct` + LoRA (r=16). The agent never
  emits a final answer — it only writes a *system prompt*.
- **LLM-under-test (frozen):** `Qwen/Qwen2.5-0.5B-Instruct`. Receives the
  agent's system prompt + the task, then generates a candidate answer.
- **Reward:** programmatic verifier (regex / `exec` / `jsonschema`) scores the
  answer. Decomposed as:
  ```
  total = correctness + 0.1 · format_bonus + brevity_penalty
  ```
- **Optimizer:** TRL 0.21 GRPO, β=0.04, T=1.0, group size G=4.
- **Dataset:** 60 train / 30 test held-out tasks across math, code, JSON.

## Submission links

- HF Space (live demo): https://huggingface.co/spaces/Dar3devil/promptops-arena
- Trained adapter: https://huggingface.co/Dar3devil/promptops-arena-agent
- Env source dataset: https://huggingface.co/datasets/Dar3devil/promptops-arena-src
- Blog: https://huggingface.co/spaces/Dar3devil/promptops-arena/blob/main/BLOG.md
- GitHub: https://github.com/Aarya01Patil/promptops_arena

## Hardware required

| GPU | Time for 300 steps · batch 8 · G=4 |
|---|---|
| H200 (141 GB) | ~25 min |
| A100-80GB     | ~45 min |
| L4 / L40S     | ~75 min |
| T4 (16 GB)    | ~2 h (lower batch needed) |
| CPU only      | not feasible (use `STEPS=2` smoke run only) |

Set `STEPS=20` in cell 6 for a quick proof-of-life run that still produces a real
(short) reward curve.

## 1 · Confirm GPU

In [ ]:
!nvidia-smi || echo 'no nvidia-smi - notebook will run on CPU and training will be very slow'

## 2 · Install pinned TRL 0.21 GRPO stack

These exact pins are what we used for the submitted run. Newer TRL versions
renamed several `GRPOConfig` fields; the training script tolerates both, but the
pins below are the safe path.

In [ ]:
!pip install -q --upgrade pip
!pip install -q \
    "trl==0.21.0" \
    "transformers==4.55.4" \
    "peft==0.15.2" \
    "accelerate==1.7.0" \
    "datasets==3.6.0" \
    "huggingface_hub>=0.25.0" \
    "jsonschema>=4.20.0" \
    "openenv-core>=0.1.0" \
    "fastapi>=0.110.0" \
    "uvicorn>=0.27.0" \
    "pydantic>=2.0.0" \
    "matplotlib>=3.7.0"

## 3 · Pull environment source from public HF dataset

The full env (verifiers, tasks, server, training script) is mirrored to a public
HF dataset, so this notebook is self-contained — no `git clone`, no auth needed.

In [ ]:
import os, sys
from huggingface_hub import snapshot_download

src_dir = snapshot_download(
    repo_id='Dar3devil/promptops-arena-src',
    repo_type='dataset',
    local_dir='/content/promptops-arena' if os.path.exists('/content') else './promptops-arena',
)
os.chdir(src_dir)
sys.path.insert(0, src_dir)
print('working dir:', os.getcwd())
print('\ntop-level files:')
!ls

## 4 · Inspect the environment

The OpenEnv environment lives in `src/envs/promptops_arena/`. Below we:

1. Load the train tasks (60 rows across math / code / JSON).
2. Construct `PromptOpsArenaEnvironment` — same class GRPO will call.
3. Run a single zero-shot prompt through it to confirm the LLM-under-test loads
   and the verifier returns a structured reward.

The first cell run is slow because it downloads `Qwen2.5-0.5B-Instruct` (~1 GB)
from the Hub. It is cached for the rest of the notebook.

In [ ]:
os.environ['PROMPTOPS_LLM_BACKEND'] = 'transformers'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from src.envs.promptops_arena.server.environment import PromptOpsArenaEnvironment
from src.envs.promptops_arena.tasks import load_tasks

tasks = load_tasks(split='train')
print(f'loaded {len(tasks)} train tasks')
by_type = {}
for t in tasks:
    by_type.setdefault(t['type'], []).append(t)
for tt, lst in by_type.items():
    print(f"  {tt:5s}: n={len(lst):2d}  example: {lst[0]['question'][:70]}")

env = PromptOpsArenaEnvironment(split='train', seed=0)
demo_task = by_type['math'][0]
demo_prompt = 'Solve this:'
res = env.execute_prompt(demo_task, demo_prompt)
print('\n--- zero-shot probe on a math task ---')
print('task:', demo_task['question'])
print('LLM-under-test output:', res['completion'][:200])
print('reward:', res['reward'])

## 5 · Reward decomposition (sanity check)

Before training, verify the reward is bounded and ungameable. We exercise three
adversarial inputs and confirm `total ≤ 0.1` unless the answer is actually
correct.

The full adversarial suite (22 tests) is in `tests/test_rewards.py`.

In [ ]:
from src.envs.promptops_arena.server.rewards import compute_reward
from src.envs.promptops_arena.verifiers import verify

math_task = by_type['math'][0]
expected = math_task['answer']
print(f"math task expects: {expected!r}")

short_sys = 'Solve this:'
long_sys  = 'You are a helpful assistant. ' * 200  # > 800 chars triggers brevity penalty

adversarial = [
    ('empty answer tags',     short_sys, '<answer></answer>'),
    ('wrong number in tags',  short_sys, '<answer>9999</answer>'),
    ('rambling no tags',      short_sys, 'lorem ipsum ' * 200),
    ('long sys prompt + wrong', long_sys, '<answer>9999</answer>'),
    ('actually correct',      short_sys, f'<answer>{expected}</answer>'),
]
print(f'\n{\"adversarial input\":28s}  {\"total\":>7s}  {\"correct\":>8s}  {\"format\":>7s}  {\"brevity\":>8s}')
print('-' * 70)
for label, sys_p, completion in adversarial:
    vres = verify(math_task, completion)
    r = compute_reward(math_task, sys_p, completion, vres)
    print(f'{label:28s}  {r[\"total\"]:+7.3f}  {r[\"correctness\"]:>8.0f}  {r[\"format\"]:>7.0f}  {r[\"brevity\"]:+8.3f}')

## 6 · GRPO training

All training logic is in `scripts/train_grpo.py`. Headline-run config:

| Knob | Value | Why |
|---|---|---|
| `--steps` | 300 | 60 train rows × ~20 epochs of effective coverage |
| `--batch` | 8 | per-device; H200 fits this comfortably |
| `--num-generations` | 4 | GRPO group size G; advantage = (r − group mean) / group std |
| `--lr` | 5e-6 | conservative for LoRA on 1.5B |
| LoRA r | 16 | applied to all attn + MLP projections |
| β (KL) | 0.04 | TRL default; keeps adapter close to base policy |
| T (sampling) | 1.0 | full-temperature exploration during rollouts |

`training_log.jsonl` records *every* reward call (one line per `(step, completion)`),
decomposed into `correctness / format / brevity` so we can plot the curve.

**Set `STEPS=20` below for a fast smoke run** that still demonstrates the
training loop and produces a (short) reward curve.

In [ ]:
STEPS = 300
BATCH = 8
NUM_GENS = 4
OUT_DIR = 'outputs/grpo-lora'
LOG_PATH = 'results/training_log.jsonl'

import shutil
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)

!python scripts/train_grpo.py \
    --steps {STEPS} \
    --batch {BATCH} \
    --num-generations {NUM_GENS} \
    --out {OUT_DIR} \
    --log {LOG_PATH}

## 7 · Plot the reward curve from this run

We plot the per-rollout `total` reward and a moving average. On a healthy run
you should see the moving average climb from ≈0.1 (formatted but wrong) toward
≈0.7–0.9 (formatted *and* correct on most train tasks).

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

rows = [json.loads(l) for l in Path(LOG_PATH).read_text().splitlines() if l.strip()]
assert rows, 'no rows in training_log.jsonl - did training fail?'

rewards     = [r['reward']['total']       for r in rows]
correctness = [r['reward']['correctness'] for r in rows]
fmt         = [r['reward']['format']      for r in rows]

win = max(1, len(rewards) // 30)
ma  = [sum(rewards[max(0, i - win):i + 1]) / max(1, min(i + 1, win))
       for i in range(len(rewards))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(rewards, alpha=0.25, label='per-rollout total')
axes[0].plot(ma, lw=2, label=f'moving avg (window={win})')
axes[0].set_xlabel('reward call # (≈ step × num_generations)')
axes[0].set_ylabel('total reward')
axes[0].set_title('GRPO reward curve - PromptOps Arena')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(correctness, alpha=0.4, label='correctness')
axes[1].plot(fmt, alpha=0.4, label='format')
axes[1].set_xlabel('reward call #')
axes[1].set_ylabel('component value (0/1)')
axes[1].set_title('Reward components over time')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
Path('docs').mkdir(exist_ok=True)
plt.savefig('docs/reward_curve.png', dpi=150)
plt.show()

print(f'rollouts logged : {len(rewards)}')
print(f'mean reward     : {sum(rewards)/len(rewards):.3f}')
print(f'fraction correct: {sum(correctness)/len(correctness):.3f}')
print(f'fraction formatted: {sum(fmt)/len(fmt):.3f}')

## 8 · Evaluate the trained adapter on the held-out test split

`scripts/eval_trained.py` loads the LoRA we just saved, generates with
`max_turns` self-correction passes, and writes a JSON summary. The submitted
headline run scored **10/12 correct, mean reward 0.917** on this same eval.

In [ ]:
!python scripts/eval_trained.py \
    --adapter {OUT_DIR} \
    --per-type 4 \
    --out results/trained_agent.json \
    --max-turns 2

import json
trained = json.load(open('results/trained_agent.json'))
print('\n=== test-split evaluation ===')
print('overall :', trained['overall'])
print('by_type :')
for tt, d in trained['by_type'].items():
    print(f"  {tt:5s}: correct {d['correct']}/{d['n']}  format {d['format']}/{d['n']}")

## 9 · Compare against baselines

Three reference policies were run on the same held-out test split with the same
frozen 0.5B LLM-under-test (results pre-computed and shipped in the env
dataset):

1. **zero-shot** — the agent always emits `"Solve this:"` (1 turn).
2. **chain-of-thought** — a hand-written CoT scaffold per task type (1 turn).
3. **untrained 1.5B agent** — same Qwen2.5-1.5B base, no LoRA, with 3
   self-correction turns (sees prev prompt + bad output, revises).

The trained agent's value-add is **per-turn efficiency**: it writes a much
better *first* prompt, which is exactly the skill GRPO was supposed to
install.

In [ ]:
import json
from pathlib import Path

files = {
    'zero-shot (1 turn)'              : 'results/baseline_zero_shot_real.json',
    'CoT (1 turn)'                    : 'results/baseline_cot_real.json',
    'untrained 1.5B agent (3 turns)'  : 'results/baseline_untrained_real.json',
    'TRAINED 1.5B agent (2 turns)'    : 'results/trained_agent.json',
}
rows = []
for label, path in files.items():
    if not Path(path).exists():
        rows.append((label, '-', '-', '-', '-'))
        continue
    d = json.load(open(path))
    o = d['overall']
    rows.append((label, o['n'], f"{o['correct']}/{o['n']}", f"{o['format']}/{o['n']}", f"{o['mean_reward']:.3f}"))

print(f'{"policy":35s}  {"n":>3s}  {"correct":>8s}  {"format":>8s}  {"mean_reward":>11s}')
print('-' * 72)
for r in rows:
    print(f'{r[0]:35s}  {str(r[1]):>3s}  {r[2]:>8s}  {r[3]:>8s}  {r[4]:>11s}')

## What this notebook proves

1. **Real GRPO loop, not a stub.** Every reward in `training_log.jsonl` came
   from the *frozen 0.5B LLM-under-test* generating an answer to a real task,
   scored by a programmatic verifier (regex / `exec` / `jsonschema`).
2. **Reward signal grounded in another model's behavior.** The agent learns how
   to *instruct*, not how to *solve* — which is what makes the skill transfer
   across math, code, and JSON.
3. **Programmatic verifiers, no reward model.** Cell 5 above demonstrates the
   reward is bounded and ungameable; the full adversarial suite
   (`tests/test_rewards.py`, 22 tests) confirms it.
4. **Reproducible from scratch.** The notebook pulls the env source from a
   public HF dataset and runs the same training script that produced the
   submitted adapter.